# Model Architecture

In [29]:
from pathlib import Path
import sys

# Local project root by default. On Kaggle, set DATA_DIR to the extracted dataset root.
DATA_DIR = Path("/kaggle/working/competition_data") if Path("/kaggle/working/competition_data/train_labels.csv").exists() else Path.cwd()
# Find the shared module in the project root, or at the Kaggle upload path.
KAGGLE_MODULE_DIR = Path("/kaggle/input/datasets/shawnleeyz/pipeline")
if KAGGLE_MODULE_DIR.exists():
    sys.path.insert(0, str(KAGGLE_MODULE_DIR))

from data_pipeline import create_dataloaders

data = create_dataloaders(data_dir=DATA_DIR)
train_loader = data.train_loader
val_loader = data.val_loader
test_loader = data.test_loader
class_to_idx = data.class_to_idx
idx_to_class = data.idx_to_class

print("Dataset root:", DATA_DIR)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Dataset root: c:\Users\yzsha\Documents\DataAndAI_project\KaggleCompetitionProject\KaggleCompetitionENGE707
Training batches: 1592
Validation batches: 398
Test batches: 921


## 1. Load pretrained ResNet18

In [30]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# Load ResNet18 pretrained on ImageNet
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## 2. Change Fully Conected Layer 

From 1000 default outputs to 8 outputs to match the amount of target class in this dataset.

In [31]:
# Change ResNet18 from 1000 ImageNet classes to 8 classes
num_classes = 8

original_features = model.fc.in_features

model.fc = nn.Linear(
    in_features=original_features,
    out_features=num_classes
)

print("Modified classification layer:", model.fc)

Modified classification layer: Linear(in_features=512, out_features=8, bias=True)


In [32]:
# Test the model with a sample batch
model.eval()

with torch.no_grad():
    sample_input = torch.randn(2, 3, 224, 224)
    output = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", output.shape)

assert output.shape == (2, 8)

print("Model output verification passed!")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 8])
Model output verification passed!


## 3. Freeze Pretrained Weights

For the initial training and validation of the model the pretrained weights would not be updated.

In [33]:
# Freeze pretrained backbone parameters
for param in model.parameters():
    param.requires_grad = False

# Replace the original classification layer
model.fc = nn.Linear(model.fc.in_features, 8)

In [34]:
# Freeze all pretrained backbone parameters; keep the existing 8-class FC layer trainable.
# Do NOT recreate model.fc here: it was already replaced in Section 2.
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

In [35]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

fc.weight
fc.bias


## 4. Configure Loss Function and Optimiser

In [36]:
import torch
import torch.nn as nn

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimiser - only update the classification layer
optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

# Training configuration
num_epochs = 10

# Select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)
print("Loss function:", criterion)
print("Optimiser:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Epochs:", num_epochs)

Device: cpu
Loss function: CrossEntropyLoss()
Optimiser: Adam
Learning rate: 0.001
Epochs: 10


## 5. Training Loop

In [37]:
# Load one batch of training images
images, labels = next(iter(train_loader))

# Move images and labels to the selected device
images = images.to(device)
labels = labels.to(device)

# Evaluation mode for this verification only
model.eval()

with torch.no_grad():
    outputs = model(images)
    loss = criterion(outputs, labels)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Label shape:", labels.shape)
print("Loss:", loss.item())

Input shape: torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 8])
Label shape: torch.Size([32])
Loss: 2.594193935394287


### 5.1 Training 1 epoch method

In [38]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    # During head-only training, preserve pretrained BatchNorm running statistics.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

### 5.2 Validation for 1 epoch method

In [39]:
def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate validation loss
            loss = criterion(outputs, labels)

            # Record loss and accuracy
            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total

### 5.3 Local one-batch smoke test (safe)

This verifies that training and validation execute locally without running full epochs. It trains a temporary copy of the model so your original model and any downloaded checkpoint remain untouched.

In [40]:
import copy

smoke_model = copy.deepcopy(model)
smoke_optimizer = torch.optim.Adam(smoke_model.fc.parameters(), lr=0.001)

one_train_batch = [next(iter(train_loader))]
one_val_batch = [next(iter(val_loader))]

smoke_train_loss, smoke_train_acc = train_one_epoch(
    smoke_model, one_train_batch, criterion, smoke_optimizer, device
)
smoke_val_loss, smoke_val_acc = validate_one_epoch(
    smoke_model, one_val_batch, criterion, device
)

print(f"One-batch training smoke test passed: loss={smoke_train_loss:.4f}, accuracy={smoke_train_acc:.2%}")
print(f"One-batch validation smoke test passed: loss={smoke_val_loss:.4f}, accuracy={smoke_val_acc:.2%}")
del smoke_model, smoke_optimizer

One-batch training smoke test passed: loss=2.3980, accuracy=0.00%
One-batch validation smoke test passed: loss=2.1753, accuracy=15.62%


In [41]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)
    loss = criterion(outputs, labels)

print("Images device:", images.device)
print("Model output:", outputs.shape)
print("Loss:", loss.item())

Images device: cpu
Model output: torch.Size([32, 8])
Loss: 2.373913288116455


### 5.3 Full Training Loop
Disabled by default, set RUN_FULL_TRAINING to true to run the full traning loop through 10 epoch.

In [42]:
# Set to True only when you intentionally want to train for all 10 epochs.
# False lets "Run All" perform setup/smoke tests without hours of CPU training.
RUN_FULL_TRAINING = False


from pathlib import Path
from IPython.display import FileLink, display
import torch
import json

num_epochs = 10

# Store training and validation results
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_accuracy = -1.0

# Save files in Kaggle's working directory
# Use a separate output folder, never overwrite your downloaded best_resnet18.pth.
# Works on Windows and Kaggle; Kaggle outputs are saved under /kaggle/working.
output_root = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
output_dir = output_root / "training_outputs"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "best_resnet18_new_run.pth"
history_path = output_dir / "training_history_new_run.json"

if RUN_FULL_TRAINING:
    for epoch in range(num_epochs):

        # Train for one epoch
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Evaluate on validation data
        val_loss, val_acc = validate_one_epoch(
            model, val_loader, criterion, device
        )

        # Record training results
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        # Display epoch results
        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2%}",
            flush=True
        )

        # Save the best model based on validation accuracy
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "class_to_idx": class_to_idx,
                    "epoch": epoch + 1,
                    "val_accuracy": val_acc
                },
                checkpoint_path
            )

            print(
                f"Best model saved! Validation accuracy: {val_acc:.2%}"
            )
            print("Checkpoint exists:", checkpoint_path.exists())
            print(
                "Checkpoint size (MB):",
                round(checkpoint_path.stat().st_size / 1024**2, 2)
            )

        # Save training history after every epoch
        with open(history_path, "w") as f:
            json.dump(history, f, indent=4)

    print(
        f"\nTraining complete. "
        f"Best validation accuracy: {best_val_accuracy:.2%}"
    )

    # Final file verification
    print("\nSaved files:")
    print("Model:", checkpoint_path)
    print("Model exists:", checkpoint_path.exists())
    print("History:", history_path)
    print("History exists:", history_path.exists())

    # Display download links
    if checkpoint_path.exists():
        display(FileLink(str(checkpoint_path)))

    if history_path.exists():
        display(FileLink(str(history_path)))

else:
    print("Full training skipped. Set RUN_FULL_TRAINING = True to run 10 epochs.")


Full training skipped. Set RUN_FULL_TRAINING = True to run 10 epochs.


## 6. Load an existing trained checkpoint for evaluation / inference

Run this only when you want to use your previously trained weights. The checkpoint should be downloaded alongside this notebook as best_resnet18.pth, or change saved_checkpoint to its path. This does not train the model. Run this after constructing the modified eight-class ResNet18 model.

In [43]:
saved_checkpoint = Path("best_resnet18.pth")

if saved_checkpoint.exists():
    checkpoint = torch.load(saved_checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("Loaded checkpoint:", saved_checkpoint)
    print("Best saved epoch:", checkpoint.get("epoch", "unknown"))
    if "val_accuracy" in checkpoint:
        print(f"Saved validation accuracy: {checkpoint['val_accuracy']:.2%}")
    assert checkpoint["class_to_idx"] == class_to_idx, "Class mapping differs from dataset!"
else:
    print("No downloaded checkpoint at", saved_checkpoint, "— skipping checkpoint loading.")


Loaded checkpoint: best_resnet18.pth
Best saved epoch: 10
Saved validation accuracy: 95.98%


## 7. Verfiy that DataLoader is functional

In [44]:
print("Test images:", len(test_loader.dataset))
print("Test batches:", len(test_loader))

images, image_ids = next(iter(test_loader))

print("Image batch shape:", images.shape)
print("Number of image IDs:", len(image_ids))
print("First five image IDs:", image_ids[:5])

Test images: 29443
Test batches: 921
Image batch shape: torch.Size([32, 3, 224, 224])
Number of image IDs: 32
First five image IDs: ('1', '2', '3', '4', '5')


## 8. Generate predictions

In [ ]:
import torch

# Use the trained model
model.eval()

predicted_classes = []
predicted_image_ids = []

# Disable gradient calculation during inference
with torch.no_grad():
    for images, image_ids in test_loader:

        images = images.to(device)

        # Generate 8 logits per image
        outputs = model(images)

        # Select the class with the highest logit
        predicted_indices = outputs.argmax(dim=1).cpu().tolist()

        # Convert numerical indices to class names
        predicted_classes.extend(
            idx_to_class[idx] for idx in predicted_indices
        )

        # Preserve the corresponding image IDs
        predicted_image_ids.extend(image_ids)

print("Total predictions:", len(predicted_classes))
print("Total image IDs:", len(predicted_image_ids))
print("First five predictions:", predicted_classes[:5])

assert len(predicted_classes) == 29443
assert len(predicted_image_ids) == 29443